# 1. Whole-tumor segmentation with a fine-tuned MedSAM2

Reproduces **Section 3.1** (the method) and **Section 4.1** (the results), namely:

| output | paper |
|---|---|
| whole-tumor Dice, IoU, precision, recall with the worst-5% column, on the cross-fitted masks | `tab:seg` |
| per-slice Dice on each held-out fold | Section 4.1, 0.86 |
| held-out Dice before any fine-tuning | Section 4.1, about 0.76 |
| the two fine-tuning inputs reaching the same held-out Dice | Section 4.1 and the Conclusions |
| how much of each reference sub-region survives inside the predicted mask | Section 4.1, 93.7% / 95.8% / 86.9% |

### The cross-fit

The masks every later notebook reads are cross-fitted on the five grading folds
(`data_cache/folds.csv`, the same `StratifiedKFold(5, shuffle=True, random_state=42)` every
grading table uses). For each fold k, MedSAM2 is fine-tuned on the cases of the other four folds
for twelve epochs, the final-epoch weights are kept with no checkpoint selection, and only the
cases of fold k are segmented with it. No case is graded from a mask predicted by a model that saw
it during fine-tuning.

### What runs here

`RUN_TRAINING` covers everything that needs a GPU: the zero-shot baseline, the five cross-fit
fine-tunes with their inference (about 100 minutes of training per fold on an Apple GPU, close to
nine hours for the five) and the two six-epoch single-split runs that compare the inputs. It is
off, and those cells were not re-run for this release: the masks and the per-fold training record
come from the original cross-fit run, whose recipe the code here implements. The training record
ships in `data_cache/crossfit_training.csv`, so the per-fold held-out Dice prints without a GPU.

The evaluation cells need the BraTS working copy and the masks in `brats_seg_cf/`, and run
whenever both are present.

Two further points the paper states as limitations:

- the box prompts come from the reference masks, so this stage still needs an annotation;
- the FLAIR, T1ce, T2 input was compared on a single 80/20 case split and was not cross-fitted;
  only the FLAIR input produced the masks the later stages read.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import medsam2, segmentation, cohort

RUN_TRAINING = False   # GPU hours; needs the BraTS copy and the MedSAM2 weights
HAVE_VOLUMES = config.NPZ_DIR.exists()
HAVE_MASKS = HAVE_VOLUMES and config.SEG_DIR.exists()
print(f"volumes present: {HAVE_VOLUMES} | cross-fitted masks present: {HAVE_MASKS}")

medsam2.hyperparameters()

## Normalization is already applied

The stored volumes have been clipped to their 1st and 99th percentiles and scaled to 8-bit
already. Applying the same transform a second time would move about 15% of voxels by one level
and the input would no longer be the one the published fine-tune saw, so the arrays are used as
they are. The cell below shows this instead of assuming it: the percentiles sit at 0 and 255, and
re-applying the transform changes nothing beyond re-rounding.

In [ ]:
if HAVE_VOLUMES:
    display(medsam2.normalization_proof(medsam2.CHANNEL_SETS["multichannel"]))

## Folds and cost

The cross-fit folds are the grading folds, and the cell below checks that case by case. A fold
trains on about 295 cases and segments the remaining 74. The timing helper measures seconds per
iteration on this machine; on an Apple GPU it came out at about 0.169 s/it, and a twelve-epoch
fold took 99 to 101 minutes.

In [ ]:
grades = cohort.load_grade_labels()
X, y, cases = cohort.build_matrix(cohort.extract_features(), grades)
folds = cohort.check_folds(cases, y)
table = folds.groupby(["fold", "grade"]).size().unstack()
table["total"] = table.sum(axis=1)
display(table)

# Uncomment to measure this machine's throughput before committing to a full run.
# medsam2.timed_batches(medsam2.CROSSFIT_CHANNELS)

## Zero-shot baseline, before any fine-tuning

MedSAM2 out of the box, prompted with the same boxes, reaches a mean Dice of **0.7694**. That is
the 0.76 the paper compares against.

In [ ]:
if RUN_TRAINING:
    channels = medsam2.CHANNEL_SETS["flair"]
    zero_shot_dir = config.DATA_ROOT / "brats_seg_zeroshot"
    medsam2.segment_all(channels, seg_out=zero_shot_dir,
                        model=medsam2.build_model("eval"))
    ids, metrics, _, _ = segmentation.scan_masks({"zero-shot": zero_shot_dir})
    print(f"zero-shot mean Dice over {len(ids)} cases: {metrics['zero-shot'][:, 0].mean():.4f}")

## The cross-fit, fold by fold

Each fold trains on the other four and segments its own cases, which land in `brats_seg_cf/`.
After training, the held-out fold is scored once at slice level, for reporting only; nothing is
selected on it. Expected test-fold slice Dice: 0.8428, 0.8595, 0.8612, 0.8673 and 0.8584, a mean
of 0.858, the paper's 0.86.

In [ ]:
if RUN_TRAINING:
    # segment_fold keeps any mask already on disk, so masks left from an earlier run would
    # silently survive a new cross-fit. Start from an empty directory.
    existing = list(config.SEG_DIR.glob("*_seg.npy")) if config.SEG_DIR.exists() else []
    assert not existing, (f"{config.SEG_DIR} already holds {len(existing)} masks; "
                          "move them away before re-running the cross-fit")
    for k in range(5):
        history, test_dice = medsam2.fine_tune_fold(k)
        display(history)
        medsam2.segment_fold(k)
    assert medsam2.check_masks(config.SEG_DIR), "mask integrity failed"

summary = medsam2.crossfit_summary()
display(summary)
print(f"mean held-out slice Dice over the five folds: "
      f"{summary.loc['mean', 'test_slice_dice']:.4f} (paper 0.86)")

## The two inputs on a single split

`flair` replicates one channel three times, which is the input the cross-fit uses.
`multichannel` puts FLAIR, T1ce and T2 in the three channels. Both are fine-tuned for six epochs
on the same 80/20 case split with the best validation epoch kept, so the comparison isolates the
input.

Expected best held-out slice Dice: **0.8642** for FLAIR and **0.8644** for the three-channel
input. The paper reports both as 0.86 and calls them equivalent.

In [ ]:
if RUN_TRAINING:
    for name in ("flair", "multichannel"):
        history, best = medsam2.fine_tune(medsam2.CHANNEL_SETS[name])
        display(history)
        print(f"{name}: best held-out slice Dice {best:.4f}")
        out = medsam2.segment_all(medsam2.CHANNEL_SETS[name])
        assert medsam2.check_masks(out), f"mask integrity failed for {name}"

## tab:seg -- the masks the rest of the pipeline works from

The cross-fitted masks, scored per case against the reference over the 368 cases that carry one.
Expected:

| Metric | Mean | Std | Worst 5% |
|---|---|---|---|
| Dice | 0.898 | 0.045 | 0.793 |
| IoU | 0.818 | 0.069 | 0.657 |
| Precision | 0.898 | 0.058 | 0.785 |
| Recall | 0.902 | 0.058 | 0.794 |

with Dice 0.897 on the high-grade cases and 0.901 on the low-grade ones. Every case is segmented
by a model that did not see it, so the cohort value differs from the held-out 0.86 only in how it
is computed: over each full volume rather than averaged over 2D slices. Precision and recall sit
close together, so the prediction is not biased towards over- or under-segmenting.

In [ ]:
if HAVE_MASKS:
    ids, metrics, recovery, pooled = segmentation.scan_masks({"predicted": config.SEG_DIR})
    grade = np.array([grades.get(c, "?") for c in ids])
    print(f"n = {len(ids)}  "
          f"(high-grade {(grade == 'HGG').sum()}, low-grade {(grade == 'LGG').sum()})")
    seg_table = segmentation.segmentation_table(metrics, "predicted", grade=grade)
    display(seg_table)

    published = {"Dice": ("0.898", "0.045", "0.793"), "IoU": ("0.818", "0.069", "0.657"),
                 "Precision": ("0.898", "0.058", "0.785"), "Recall": ("0.902", "0.058", "0.794"),
                 "Dice, high-grade": ("0.897", None, None), "Dice, low-grade": ("0.901", None, None)}
    from src import tables
    check = tables.check_printed(seg_table, {
        row: {c: v for c, v in zip(["Mean", "Std", "Worst 5%"], values) if v}
        for row, values in published.items()})
    display(check)
    print("all cells match the paper:", bool(check["match"].all()))
    display(segmentation.worst_cases(ids, metrics, "predicted", grade=grade))

## Where the missing tissue is

The errors are not spread evenly. The predicted mask holds **93.7%** of the reference enhancing
tumor and **95.8%** of the necrotic core, but only **86.9%** of the edema. Almost everything the
segmentation misses is edema at the outer margin, and the enhancing tumor and necrotic core, which
is where the grade is read from, are nearly always inside the mask. That is the first of the two
reasons the substitution in notebook 4 changes so little.

The weakest case in the cohort, BraTS20_Training_087 at a Dice of 0.71, still keeps all of the
enhancing tumor and 89% of the necrotic core while missing 57% of the edema.

In [ ]:
if HAVE_MASKS:
    display(segmentation.recovery_table(recovery, pooled, ["predicted"]))
    print(segmentation.case_recovery("BraTS20_Training_087"))

## The two inputs side by side

The masks of the second input come from its single-split fine-tune, so for the 295 cases of that
split they are in-sample, while the FLAIR masks are cross-fitted. The cohort-level metrics below
are therefore not a like-for-like comparison of the inputs; the held-out slice Dice above is.
Grading from either set of masks gives an AUC of 0.93, and the numbers reported in the paper come
from the cross-fitted FLAIR masks.

In [ ]:
_, mc_ckpt, mc_dir, mc_cache = medsam2.run_paths(medsam2.CHANNEL_SETS["multichannel"])
if HAVE_MASKS and mc_dir.exists():
    keys = ["FLAIR, cross-fitted", "FLAIR, T1ce, T2, single split"]
    ids, metrics, recovery, pooled = segmentation.scan_masks(
        {keys[0]: config.SEG_DIR, keys[1]: mc_dir})
    display(segmentation.segmentation_table(metrics, keys[0]))
    display(segmentation.segmentation_table(metrics, keys[1]))
    display(segmentation.recovery_table(recovery, pooled, keys))

In [ ]:
from src import tables

X_flair, y, cases_flair = cohort.build_matrix(cohort.extract_features(), grades)
feat_mc = cohort.extract_features(seg_dir=mc_dir, cache=mc_cache)
X_mc, y_mc, cases_mc = cohort.build_matrix(feat_mc, grades)
assert cases_flair == cases_mc, "case order must match or the folds differ"

side_by_side = pd.DataFrame({
    "FLAIR masks, cross-fitted": tables.table_clf(X_flair, y)["AUC"],
    "FLAIR, T1ce, T2 masks, single split": tables.table_clf(X_mc, y_mc)["AUC"],
})
side_by_side["difference"] = (side_by_side.iloc[:, 1] - side_by_side.iloc[:, 0]).round(4)
display(side_by_side)
print("RBF SVM, both at two decimals:",
      [f"{v:.2f}" for v in side_by_side.loc["SVM (RBF)"].iloc[:2]], "(paper 0.93)")